# Эксперименты с кандидатогенерацией

Схема валидации, простой TF-IDF бейзлайн и сигналы, которые добавлялись по одному: локация, символьные n-граммы, Word2Vec, BM25. Метрика — Recall@50 на локальной валидации.

Итоговый пайплайн вынесен в `src/avito_cg`, его результаты — в README.

In [ ]:
from pathlib import Path

import bm25s
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

DATA_DIR = Path("../dataset")
train = pd.read_parquet(DATA_DIR / "train.parquet")

print("Загружен train:", train.shape)

Загружен train: (497673, 19)


## Первый быстрый замер

До нормальной валидации — BM25 (`bm25s`) по полному тексту объявления на 1000 случайных запросах, сплит 80/20 по тексту запроса. Просто чтобы понять порядок цифр.

In [ ]:
QUERY_COLS = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]

ITEM_TEXT_COLS = [
    "item_title_raw",
    "item_description_raw",
    "item_infm_params_text",
]

def join_text(df, columns):
    return (
        df[columns]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

def make_query_key(df):
    return (
        df[QUERY_COLS]
        .fillna("")
        .astype(str)
        .agg("\x1f".join, axis=1)
    )

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
_, val_idx = next(
    splitter.split(train, groups=train["search_query"].fillna(""))
)

val_pairs = train.iloc[val_idx].copy()
val_pairs["query_key"] = make_query_key(val_pairs)

val_queries = (
    val_pairs[QUERY_COLS]
    .drop_duplicates()
    .reset_index(drop=True)
)
val_queries["query_key"] = make_query_key(val_queries)

val_sample = (
    val_queries.sample(n=min(1000, len(val_queries)), random_state=42)
    .reset_index(drop=True)
)

catalog = (
    train[["item_id"] + ITEM_TEXT_COLS]
    .drop_duplicates("item_id")
    .reset_index(drop=True)
)
item_ids = catalog["item_id"].astype(str).to_numpy()

doc_texts = join_text(catalog, ITEM_TEXT_COLS).tolist()
query_texts = join_text(
    val_sample,
    ["search_query", "search_infm_params_text"],
).tolist()

doc_tokens = bm25s.tokenize(doc_texts)
retriever = bm25s.BM25(method="lucene")
retriever.index(doc_tokens)

query_tokens = bm25s.tokenize(query_texts)
results, scores = retriever.retrieve(query_tokens, k=50)

relevant_by_query = (
    val_pairs.groupby("query_key")["item_id"]
    .agg(lambda s: set(s.astype(str)))
    .to_dict()
)

recalls = []

for row, query in val_sample.iterrows():
    query_key = query["query_key"]
    predicted_ids = set(item_ids[results[row]])
    relevant_ids = relevant_by_query[query_key]
    recalls.append(len(predicted_ids & relevant_ids) / len(relevant_ids))

print(f"Recall@50: {np.mean(recalls):.4f}")

Recall@50: 0.1897


In [ ]:
recall_report = pd.DataFrame({
    "query_key": val_sample["query_key"].astype(str),
    "recall": recalls,
})

print("Медиана Recall@50:", recall_report["recall"].median())
print("Доля запросов с Recall@50 = 0:",
      (recall_report["recall"] == 0).mean())

display(
    recall_report.sort_values("recall")
)

Медиана Recall@50: 0.0
Доля запросов с Recall@50 = 0: 0.789


,query_key,recall
0,рекламный ролик 642320 0 114,0.0
50,адвокат по гражданским делам 637640 0 Кто оказ...,0.0
49,ремонт входных дверей 641780 0 Вид услуги Ремо...,0.0
48,"шугаринг женский 653420 0 Вид услуги Красота, ...",0.0
47,страхование авто 653410 0 Вид услуги 114,0.0
45,вспахать землю 653950 0 Тип услуги Земляные ра...,0.0
43,установка кондиционеров монтаж сплит систем 64...,0.0
40,няня для собаки 638330 0 Вид услуги Уход за жи...,0.0
39,"аренда прицепа 621630 0 Вид услуги Автосервис,...",0.0
38,прокат квадроциклов 640650 0 Аренда авто Аренд...,0.0


Recall@50 ≈ 0.19, у 79% запросов не найдено ни одного правильного объявления. Есть куда расти.

## Схема валидации

- удаляю полные дубли строк;
- делю по тексту запроса (`GroupShuffleSplit`, 10%): поиск в основном текстовый, и один и тот же текст в train и val завысил бы метрику;
- корпус для поиска — все объявления train: с корпусом бенчмарка пересечение слишком маленькое.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

search_cols = ['search_query', 'search_location_id', 'search_is_delivery_search',
               'search_infm_params_text', 'search_category']
full_key = search_cols + ['item_id']

before = len(train)
train_dedup = train.drop_duplicates(subset=full_key).reset_index(drop=True)
print(f"Было строк: {before}, стало: {len(train_dedup)}, удалено: {before - len(train_dedup)}")

gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_idx, val_idx = next(gss.split(train_dedup, groups=train_dedup['search_query']))

train_local = train_dedup.iloc[train_idx].reset_index(drop=True)
val_local = train_dedup.iloc[val_idx].reset_index(drop=True)

print(f"train_local: {len(train_local)} строк, {train_local['search_query'].nunique()} уникальных запросов")
print(f"val_local:   {len(val_local)} строк, {val_local['search_query'].nunique()} уникальных запросов")

overlap = set(train_local['search_query']) & set(val_local['search_query'])
print("Общих текстов в train и val:", len(overlap))

Было строк: 497673, стало: 467054, удалено: 30619
train_local: 428429 строк, 67076 уникальных запросов
val_local:   38625 строк, 7453 уникальных запросов
Общих текстов в train и val: 0


Ground truth: для каждой комбинации полей `search_*` из val — множество выбранных объявлений. В среднем 1.2 объявления на запрос, максимум 91.

In [ ]:
query_key_cols = ['search_query', 'search_location_id', 'search_is_delivery_search',
                   'search_infm_params_text', 'search_category']

val_ground_truth = (
    val_local.groupby(query_key_cols)['item_id']
    .apply(lambda x: set(x))
    .reset_index(name='relevant_items')
)

print("Запросов в val:", len(val_ground_truth))
print(val_ground_truth['relevant_items'].apply(len).describe())

local_corpus = train_dedup.drop_duplicates(subset='item_id').reset_index(drop=True)
print("Объявлений в корпусе:", len(local_corpus))

all_relevant = set().union(*val_ground_truth['relevant_items'])
missing = all_relevant - set(local_corpus['item_id'])
print("Релевантных объявлений нет в корпусе:", len(missing))

Запросов в val: 31673
count    31673.000000
mean         1.219493
std          1.123922
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         91.000000
Name: relevant_items, dtype: float64
Объявлений в корпусе: 344825
Релевантных объявлений нет в корпусе: 0


Метрика и две проверки: идеальное предсказание даёт почти 1 (не ровно 1 из-за запросов, где релевантных больше 50), случайное — около нуля.

In [ ]:
def recall_at_50(predictions: dict, ground_truth: dict, k: int = 50) -> float:
    """
    predictions: dict {query_key: [item_id, ...]} — упорядоченный или нет список кандидатов (не важно)
    ground_truth: dict {query_key: set(item_id, ...)} — релевантные item_id
    Возвращает средний Recall@k по всем запросам из ground_truth.
    """
    recalls = []
    for query_key, relevant in ground_truth.items():
        candidates = set(predictions.get(query_key, [])[:k])
        if len(relevant) == 0:
            continue
        hit = len(candidates & relevant) / len(relevant)
        recalls.append(hit)
    return sum(recalls) / len(recalls)

dummy_gt = {k: v for k, v in zip(val_ground_truth[query_key_cols].apply(tuple, axis=1),
                                    val_ground_truth['relevant_items'])}
dummy_pred_perfect = {k: list(v) for k, v in dummy_gt.items()}
print(f"Идеальное предсказание: {recall_at_50(dummy_pred_perfect, dummy_gt):.4f}")

import random
random.seed(42)
corpus_ids = local_corpus['item_id'].tolist()
dummy_pred_random = {k: random.sample(corpus_ids, 50) for k in dummy_gt.keys()}
print(f"Случайное предсказание: {recall_at_50(dummy_pred_random, dummy_gt):.4f}")

Идеальное предсказание: 1.0000
Случайное предсказание: 0.0002


## TF-IDF по заголовку

Нижний регистр, только буквы и цифры, TF-IDF по словам заголовка.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import re

def simple_preprocess(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^а-яёa-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

corpus_texts = local_corpus['item_title_raw'].apply(simple_preprocess).tolist()
corpus_ids = local_corpus['item_id'].tolist()

val_queries_df = val_local.drop_duplicates(subset=query_key_cols)[query_key_cols].reset_index(drop=True)
query_texts = val_queries_df['search_query'].apply(simple_preprocess).tolist()

print("Объявление:", corpus_texts[0])
print("Запрос:", query_texts[0])

vectorizer = TfidfVectorizer(min_df=2)
corpus_matrix = vectorizer.fit_transform(corpus_texts)
query_matrix = vectorizer.transform(query_texts)

print("Словарь:", len(vectorizer.vocabulary_))

Объявление: скупка б у техники
Запрос: скупка телевизоров
Словарь: 25346


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

K = 50
batch_size = 500

n_queries = query_matrix.shape[0]
top_k_indices = np.zeros((n_queries, K), dtype=np.int64)

for start in range(0, n_queries, batch_size):
    end = min(start + batch_size, n_queries)
    batch = query_matrix[start:end]

    sims = cosine_similarity(batch, corpus_matrix)

    part = np.argpartition(-sims, K, axis=1)[:, :K]

    row_indices = np.arange(sims.shape[0])[:, None]
    part_sims = sims[row_indices, part]
    order_within = np.argsort(-part_sims, axis=1)
    sorted_part = part[row_indices, order_within]

    top_k_indices[start:end] = sorted_part

corpus_ids_arr = np.array(corpus_ids)
predictions_tfidf = {}
for i, row in val_queries_df.iterrows():
    query_key = tuple(row[c] for c in query_key_cols)
    item_indices = top_k_indices[i]
    predictions_tfidf[query_key] = corpus_ids_arr[item_indices].tolist()

In [ ]:
recall_tfidf_title = recall_at_50(predictions_tfidf, dummy_gt)
print(f"Recall@50: {recall_tfidf_title:.4f}")

Recall@50: 0.1546


## Локация

0.155 — слабо. Проверяю, насколько часто совпадает локация запроса и объявления.

In [ ]:
match_rate = (train_dedup['search_location_id'] == train_dedup['item_location_id']).mean()
print(f"Доля пар, где search_location_id == item_location_id: {match_rate:.4f}")

for delivery_flag in [0, 1]:
    subset = train_dedup[train_dedup['search_is_delivery_search'] == delivery_flag]
    if len(subset) > 0:
        mr = (subset['search_location_id'] == subset['item_location_id']).mean()
        print(f"  search_is_delivery_search={delivery_flag}: n={len(subset)}, match_rate={mr:.4f}")

mismatch = train_dedup[train_dedup['search_location_id'] != train_dedup['item_location_id']]
print(f"\nПримеры несовпадений ({len(mismatch)} из {len(train_dedup)}):")
print(mismatch[['search_query', 'search_location_id', 'item_location_id', 'search_is_delivery_search']].head(10))

print("\nУникальных search_location_id:", train_dedup['search_location_id'].nunique())
print("Уникальных item_location_id:", train_dedup['item_location_id'].nunique())

Доля пар, где search_location_id == item_location_id: 0.8220
  search_is_delivery_search=0: n=467042, match_rate=0.8220
  search_is_delivery_search=1: n=12, match_rate=0.5000

Примеры несовпадений (83150 из 467054):
                      search_query  search_location_id  item_location_id  \
3   изготовление госномера на авто              634670            633570   
5                    сборка мебели              639550            637770   
6               наращивание ногтей              107620            637640   
19               пополнение alipay              621540            637640   
20                    хромирование              632660            633540   
26                      липолитики              107620            637640   
29                  авто под выкуп              629990            630020   
33                      муж на час              623845            653271   
39                 кузовные работы              632660            633540   
61        ремонт кондици

Совпадение в 82% пар. Добавляю к сходству бонус за совпадение локации. Именно бонус, а не фильтр: 18% правильных ответов находятся в другом городе, фильтр отрезал бы их совсем.

In [ ]:
item_locations_arr = local_corpus['item_location_id'].values

LOCATION_BOOST = 0.15

top_k_indices_boosted = np.zeros((n_queries, K), dtype=np.int64)

for start in range(0, n_queries, batch_size):
    end = min(start + batch_size, n_queries)
    batch = query_matrix[start:end]
    batch_locations = val_queries_df['search_location_id'].values[start:end]

    sims = cosine_similarity(batch, corpus_matrix)

    location_match = (batch_locations[:, None] == item_locations_arr[None, :])
    sims = sims + LOCATION_BOOST * location_match

    part = np.argpartition(-sims, K, axis=1)[:, :K]
    row_indices = np.arange(sims.shape[0])[:, None]
    part_sims = sims[row_indices, part]
    order_within = np.argsort(-part_sims, axis=1)
    sorted_part = part[row_indices, order_within]

    top_k_indices_boosted[start:end] = sorted_part

predictions_tfidf_boosted = {}
for i, row in val_queries_df.iterrows():
    query_key = tuple(row[c] for c in query_key_cols)
    predictions_tfidf_boosted[query_key] = corpus_ids_arr[top_k_indices_boosted[i]].tolist()

recall_boosted = recall_at_50(predictions_tfidf_boosted, dummy_gt)
print(f"Recall@50 с бонусом {LOCATION_BOOST}: {recall_boosted:.4f}")
print(f"Recall@50 без бонуса: {recall_tfidf_title:.4f}")

Recall@50 с бонусом 0.15: 0.2605
Recall@50 без бонуса: 0.1546


Какие запросы всё ещё проваливаются:

In [ ]:
per_query_recall_boosted = {}
for query_key, relevant in dummy_gt.items():
    candidates = set(predictions_tfidf_boosted.get(query_key, []))
    hit = len(candidates & relevant) / len(relevant)
    per_query_recall_boosted[query_key] = hit

zero_recall_boosted = [k for k, v in per_query_recall_boosted.items() if v == 0.0]
print(f"Запросов с recall=0: {len(zero_recall_boosted)} из {len(per_query_recall_boosted)} "
      f"({len(zero_recall_boosted)/len(per_query_recall_boosted)*100:.1f}%)")

import random
random.seed(0)
sample_fails = random.sample(zero_recall_boosted, 15)

for key in sample_fails:
    query_text, s_loc = key[0], key[1]
    relevant_ids = dummy_gt[key]
    predicted_ids = predictions_tfidf_boosted[key][:5]

    relevant_items = local_corpus[local_corpus['item_id'].isin(relevant_ids)][['item_title_raw', 'item_location_id']]
    predicted_titles = [local_corpus[local_corpus['item_id'] == pid]['item_title_raw'].values[0]
                         for pid in predicted_ids]

    print(f"\nЗапрос: '{query_text}' (search_location_id={s_loc})")
    print(f"  Должно быть найдено:\n{relevant_items.to_string(index=False)}")
    print(f"  Топ-5 предсказано: {predicted_titles}")

Запросов с recall=0: 22616 из 31673 (71.4%)

Запрос: 'прицеп в аренду' (search_location_id=621630)
  Должно быть найдено:
          item_title_raw  item_location_id
Аренда легкового прицепа            621630
Аренда легковых прицепов            621630
  Топ-5 предсказано: ['Прицеп в аренду', 'Прицеп в аренду', 'Прицеп в аренду', 'Прицеп в аренду', 'Прицеп в аренду']

Запрос: 'ремонт iphone' (search_location_id=654170)
  Должно быть найдено:
                          item_title_raw  item_location_id
Ремонт телефонов iPhone / iPad / Android            654170
  Топ-5 предсказано: ['Ремонт iPhone', 'Ремонт iPhone', 'Ремонт iPhone', 'Ремонт iPhone', 'Ремонт iPhone']

Запрос: 'аренда камаз' (search_location_id=639280)
  Должно быть найдено:
        item_title_raw  item_location_id
Доставка щебня и песка            639280
  Топ-5 предсказано: ['Ремонт Камаз', 'Услуги и аренда манипулятора камаз', 'Грузоперевозки камаз', 'Аренда самосвала камаз', 'Аренда самосвала камаз']

Запрос: 'межевание' (

Перебираю величину бонуса.

In [ ]:
def evaluate_boost(boost_value, query_matrix, corpus_matrix, val_queries_df,
                    item_locations_arr, corpus_ids_arr, dummy_gt, batch_size=500, K=50):
    n_queries = query_matrix.shape[0]
    top_k_indices = np.zeros((n_queries, K), dtype=np.int64)

    for start in range(0, n_queries, batch_size):
        end = min(start + batch_size, n_queries)
        batch = query_matrix[start:end]
        batch_locations = val_queries_df['search_location_id'].values[start:end]

        sims = cosine_similarity(batch, corpus_matrix)
        location_match = (batch_locations[:, None] == item_locations_arr[None, :])
        sims = sims + boost_value * location_match

        part = np.argpartition(-sims, K, axis=1)[:, :K]
        row_indices = np.arange(sims.shape[0])[:, None]
        part_sims = sims[row_indices, part]
        order_within = np.argsort(-part_sims, axis=1)
        sorted_part = part[row_indices, order_within]
        top_k_indices[start:end] = sorted_part

    predictions = {}
    for i, row in val_queries_df.iterrows():
        query_key = tuple(row[c] for c in query_key_cols)
        predictions[query_key] = corpus_ids_arr[top_k_indices[i]].tolist()

    return recall_at_50(predictions, dummy_gt)

for boost in [0.05, 0.1, 0.15, 0.25, 0.4, 0.6]:
    r = evaluate_boost(boost, query_matrix, corpus_matrix, val_queries_df,
                        item_locations_arr, corpus_ids_arr, dummy_gt)
    print(f"boost={boost}: Recall@50 = {r:.4f}")

boost=0.05: Recall@50 = 0.2000
boost=0.1: Recall@50 = 0.2281
boost=0.15: Recall@50 = 0.2605
boost=0.25: Recall@50 = 0.3253
boost=0.4: Recall@50 = 0.4111
boost=0.6: Recall@50 = 0.4827


In [ ]:
for boost in [0.8, 1.0, 1.5, 2.0, 3.0]:
    r = evaluate_boost(boost, query_matrix, corpus_matrix, val_queries_df,
                        item_locations_arr, corpus_ids_arr, dummy_gt)
    print(f"boost={boost}: Recall@50 = {r:.4f}")

boost=0.8: Recall@50 = 0.5179
boost=1.0: Recall@50 = 0.5263
boost=1.5: Recall@50 = 0.5280
boost=2.0: Recall@50 = 0.5280
boost=3.0: Recall@50 = 0.5280


Рост останавливается на 1.5: при таком бонусе объявления своей локации почти всегда выше чужих, и текст ранжирует уже внутри локации. Фиксирую 1.5.

In [ ]:
LOCATION_BOOST_FINAL = 1.5

top_k_indices_final = np.zeros((n_queries, K), dtype=np.int64)

for start in range(0, n_queries, batch_size):
    end = min(start + batch_size, n_queries)
    batch = query_matrix[start:end]
    batch_locations = val_queries_df['search_location_id'].values[start:end]

    sims = cosine_similarity(batch, corpus_matrix)
    location_match = (batch_locations[:, None] == item_locations_arr[None, :])
    sims = sims + LOCATION_BOOST_FINAL * location_match

    part = np.argpartition(-sims, K, axis=1)[:, :K]
    row_indices = np.arange(sims.shape[0])[:, None]
    part_sims = sims[row_indices, part]
    order_within = np.argsort(-part_sims, axis=1)
    sorted_part = part[row_indices, order_within]
    top_k_indices_final[start:end] = sorted_part

predictions_final = {}
for i, row in val_queries_df.iterrows():
    query_key = tuple(row[c] for c in query_key_cols)
    predictions_final[query_key] = corpus_ids_arr[top_k_indices_final[i]].tolist()

recall_final = recall_at_50(predictions_final, dummy_gt)
print(f"Recall@50: {recall_final:.4f}")

Recall@50: 0.5280


## Разбор ошибок

Делю запросы с нулевым recall на две группы: правильное объявление было в той же локации (не справился текст) и правильное объявление в другой локации.

In [ ]:
per_query_recall_final = {}
for query_key, relevant in dummy_gt.items():
    candidates = set(predictions_final.get(query_key, []))
    hit = len(candidates & relevant) / len(relevant)
    per_query_recall_final[query_key] = hit

zero_recall_final = [k for k, v in per_query_recall_final.items() if v == 0.0]
print(f"Запросов с recall=0: {len(zero_recall_final)} из {len(per_query_recall_final)} "
      f"({len(zero_recall_final)/len(per_query_recall_final)*100:.1f}%)")

fail_location_match = []
fail_location_mismatch = []

for key in zero_recall_final:
    query_text, s_loc = key[0], key[1]
    relevant_ids = dummy_gt[key]
    relevant_locs = local_corpus[local_corpus['item_id'].isin(relevant_ids)]['item_location_id'].tolist()
    if s_loc in relevant_locs:
        fail_location_match.append(key)
    else:
        fail_location_mismatch.append(key)

print(f"\nПравильное объявление в той же локации: {len(fail_location_match)}")
print(f"Правильное объявление в другой локации: {len(fail_location_mismatch)}")

import random
random.seed(1)
sample_fails = random.sample(fail_location_match, min(15, len(fail_location_match)))

for key in sample_fails:
    query_text, s_loc = key[0], key[1]
    relevant_ids = dummy_gt[key]
    predicted_ids = predictions_final[key][:5]

    relevant_titles = local_corpus[local_corpus['item_id'].isin(relevant_ids)]['item_title_raw'].tolist()
    predicted_titles = [local_corpus[local_corpus['item_id'] == pid]['item_title_raw'].values[0]
                         for pid in predicted_ids]

    print(f"\nЗапрос: '{query_text}'")
    print(f"  Должно быть найдено: {relevant_titles}")
    print(f"  Топ-5 предсказано: {predicted_titles}")

Запросов с recall=0: 14236 из 31673 (44.9%)

Правильное объявление в той же локации: 8822
Правильное объявление в другой локации: 5414

Запрос: 'заливка бетонных дорожек'
  Должно быть найдено: ['Бетонные работы']
  Топ-5 предсказано: ['Заливка фундамента', 'Аренда оборудования для бетонных работ', 'Шлифовка бетонных полов в Уфе', 'Фундамент под ключ / Заливка фундамента', 'Заливка фундамента бетонные работы под ключ']

Запрос: 'бонбоньерки на свадьбу'
  Должно быть найдено: ['Комплименты гостям мёд 50 мл / бонбоньерки']
  Топ-5 предсказано: ['Фотограф на свадьбу / видеограф на свадьбу и никах', 'Фотограф и видеограф на свадьбу', 'Видеограф, фотограф на свадьбу', 'Пригласительные на свадьбу', 'Пригласительные на свадьбу']

Запрос: 'отделочные работы'
  Должно быть найдено: ['Ремонт квартир и офисов под ключ']
  Топ-5 предсказано: ['Строительные работы', 'Электромонтажные работы', 'Сантехнические работы', 'Бетонные работы, заливка фундамента', 'Строительные и монтажные работы']

Запрос:

Текстовые ошибки бывают нескольких видов:

- словоформы: «телевизор» и «телевизора» для TF-IDF разные слова;
- слишком короткий заголовок у правильного объявления («Бетонные работы» проигрывает более длинным);
- синонимы без общих слов («отделочные работы» → «Ремонт квартир под ключ»);
- общие слова вроде «услуги» или «ГОСТ», которые дают ложное сходство.

Ошибки из-за другой локации. Часто текстово найдено ровно то, что нужно, но в другом городе: в корпусе много почти одинаковых объявлений в разных городах. Часть таких ошибок — ограничение офлайн-оценки, а не модели.

In [ ]:
import random
random.seed(2)
sample_mismatch = random.sample(fail_location_mismatch, 15)

for key in sample_mismatch:
    query_text, s_loc = key[0], key[1]
    relevant_ids = dummy_gt[key]
    predicted_ids = predictions_final[key][:5]

    relevant_info = local_corpus[local_corpus['item_id'].isin(relevant_ids)][
        ['item_title_raw', 'item_location_id']
    ]
    predicted_titles = [local_corpus[local_corpus['item_id'] == pid]['item_title_raw'].values[0]
                         for pid in predicted_ids]

    print(f"\nЗапрос: '{query_text}' (search_location_id={s_loc})")
    print(f"  Должно быть найдено:\n{relevant_info.to_string(index=False)}")
    print(f"  Топ-5 предсказано: {predicted_titles}")

Запрос: 'аренда прицепа для легкового автомобиля' (search_location_id=107620)
  Должно быть найдено:
          item_title_raw  item_location_id
Аренда легковых прицепов            639640
  Топ-5 предсказано: ['Аренда прицепа для легкового автомобиля', 'Аренда прицепа для легкового автомобиля', 'Аренда прицепа для легкового автомобиля', 'Аренда прицепа для легкового автомобиля', 'Аренда прицепа для легкового автомобиля']

Запрос: 'вибротрамбовка' (search_location_id=651820)
  Должно быть найдено:
           item_title_raw  item_location_id
Аренда/прокат инструмента            652100
  Топ-5 предсказано: ['Прокат Кроссовых Мотоциклов', 'Ремонт кондиционеров и сплит систем', 'Сварочные работы', 'Выкачка сливных ям и туалетов', 'Вибротрамбовка шлёпнога']

Запрос: 'броне пленка на авто' (search_location_id=640360)
  Должно быть найдено:
                             item_title_raw  item_location_id
Оклейка автомобиля бронепленкой / Детейлинг            640350
  Топ-5 предсказано: ['Покраска 

## Близкие локации

Гипотеза: близкие числовые `location_id` означают близкие места. Смотрю на разницу ID в несовпадающих парах.

In [ ]:
train_dedup['loc_diff'] = (train_dedup['search_location_id'] - train_dedup['item_location_id']).abs()

exact_match = (train_dedup['loc_diff'] == 0).mean()
print(f"Точное совпадение (diff=0): {exact_match:.4f}")

for threshold in [50, 100, 500, 1000, 5000,10000]:
    coverage = (train_dedup['loc_diff'] <= threshold).mean()
    print(f"Покрытие при |diff| <= {threshold}: {coverage:.4f} (прирост над точным: {coverage - exact_match:+.4f})")

mismatched = train_dedup[train_dedup['loc_diff'] > 0]
print("\nРаспределение loc_diff среди несовпадающих пар:")
print(mismatched['loc_diff'].describe())
print("\nПерцентили:", mismatched['loc_diff'].quantile([0.1, 0.25, 0.5, 0.75, 0.9]).to_dict())

Точное совпадение (diff=0): 0.8220
Покрытие при |diff| <= 50: 0.8391 (прирост над точным: +0.0171)
Покрытие при |diff| <= 100: 0.8497 (прирост над точным: +0.0278)
Покрытие при |diff| <= 500: 0.8976 (прирост над точным: +0.0756)
Покрытие при |diff| <= 1000: 0.9162 (прирост над точным: +0.0942)
Покрытие при |diff| <= 5000: 0.9295 (прирост над точным: +0.1075)
Покрытие при |diff| <= 10000: 0.9341 (прирост над точным: +0.1121)

Распределение loc_diff среди несовпадающих пар:
count     83150.000000
mean     124525.193013
std      220816.371801
min           1.000000
25%         210.000000
50%         880.000000
75%       31700.000000
max      797131.000000
Name: loc_diff, dtype: float64

Перцентили: {0.1: 60.0, 0.25: 210.0, 0.5: 880.0, 0.75: 31700.0, 0.9: 530910.0}


Медиана разницы 880, но хвост до 800 тыс. Похоже, соседние ID часто соседние места, но шкала нигде не описана. Пробую второй бонус за |разницу ID| ≤ порога.

In [ ]:
LOCATION_BOOST_EXACT = 1.5

def evaluate_two_level_boost(exact_boost, near_boost, near_threshold,
                               query_matrix, corpus_matrix, val_queries_df,
                               item_locations_arr, corpus_ids_arr, dummy_gt,
                               batch_size=100, K=50):
    n_queries = query_matrix.shape[0]
    top_k_indices = np.zeros((n_queries, K), dtype=np.int64)

    item_locations_i32 = item_locations_arr.astype(np.int32)

    for start in range(0, n_queries, batch_size):
        end = min(start + batch_size, n_queries)
        batch = query_matrix[start:end]
        batch_locations = val_queries_df['search_location_id'].values[start:end].astype(np.int32)

        sims = cosine_similarity(batch, corpus_matrix).astype(np.float32)

        exact_match = (batch_locations[:, None] == item_locations_i32[None, :])
        sims += exact_boost * exact_match
        del exact_match

        diff = np.abs(batch_locations[:, None] - item_locations_i32[None, :])
        near_match = (diff > 0) & (diff <= near_threshold)
        del diff

        sims += near_boost * near_match
        del near_match

        part = np.argpartition(-sims, K, axis=1)[:, :K]
        row_indices = np.arange(sims.shape[0])[:, None]
        part_sims = sims[row_indices, part]
        order_within = np.argsort(-part_sims, axis=1)
        sorted_part = part[row_indices, order_within]
        top_k_indices[start:end] = sorted_part

        del sims

    predictions = {}
    for i, row in val_queries_df.iterrows():
        query_key = tuple(row[c] for c in query_key_cols)
        predictions[query_key] = corpus_ids_arr[top_k_indices[i]].tolist()

    return recall_at_50(predictions, dummy_gt)

for near_thresh in [500, 1000, 5000]:
    for near_b in [0.1, 0.3, 0.5]:
        r = evaluate_two_level_boost(LOCATION_BOOST_EXACT, near_b, near_thresh,
                                       query_matrix, corpus_matrix, val_queries_df,
                                       item_locations_arr, corpus_ids_arr, dummy_gt,
                                       batch_size=100)
        print(f"near_threshold={near_thresh}, near_boost={near_b}: Recall@50 = {r:.4f}")

near_threshold=500, near_boost=0.1: Recall@50 = 0.5355
near_threshold=500, near_boost=0.3: Recall@50 = 0.5438
near_threshold=500, near_boost=0.5: Recall@50 = 0.5481
near_threshold=1000, near_boost=0.1: Recall@50 = 0.5361
near_threshold=1000, near_boost=0.3: Recall@50 = 0.5461
near_threshold=1000, near_boost=0.5: Recall@50 = 0.5494
near_threshold=5000, near_boost=0.1: Recall@50 = 0.5333
near_threshold=5000, near_boost=0.3: Recall@50 = 0.5386
near_threshold=5000, near_boost=0.5: Recall@50 = 0.5376


Лучше всего порог 1000 и бонус 0.5: 0.5494. В пайплайне эту эвристику потом заменило настоящее расстояние между центроидами локаций по координатам объявлений.

## Recall@200

Сколько правильных объявлений в топ-200 — это потолок для возможного переранжирования.

In [ ]:
import gc
gc.collect()

K200 = 200
batch_size_200 = 100

top_k_indices_200 = np.zeros((n_queries, K200), dtype=np.int64)

for start in range(0, n_queries, batch_size_200):
    end = min(start + batch_size_200, n_queries)
    batch = query_matrix[start:end]
    batch_locations = val_queries_df['search_location_id'].values[start:end]

    sims = cosine_similarity(batch, corpus_matrix).astype(np.float32)
    location_match = (batch_locations[:, None] == item_locations_arr[None, :])
    sims += LOCATION_BOOST_FINAL * location_match
    del location_match

    part = np.argpartition(-sims, K200, axis=1)[:, :K200]
    row_indices = np.arange(sims.shape[0])[:, None]
    part_sims = sims[row_indices, part]
    order_within = np.argsort(-part_sims, axis=1)
    sorted_part = part[row_indices, order_within]
    top_k_indices_200[start:end] = sorted_part

    del sims

predictions_200 = {}
for i, row in val_queries_df.iterrows():
    query_key = tuple(row[c] for c in query_key_cols)
    predictions_200[query_key] = corpus_ids_arr[top_k_indices_200[i]].tolist()

recall_200 = recall_at_50(predictions_200, dummy_gt, k=200)
recall_50_check = recall_at_50(predictions_200, dummy_gt, k=50)

print(f"Recall@50:  {recall_50_check:.4f}")
print(f"Recall@200: {recall_200:.4f}")
print(f"Разница:    {recall_200 - recall_50_check:+.4f}")

Recall@50:  0.5280
Recall@200: 0.6313
Разница:    +0.1034


Между топ-50 и топ-200 около 0.10. Реранкер пока рано: сначала нужно расширять сами каналы поиска.

## Символьные n-граммы

От словоформ помогают символьные n-граммы внутри слов.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

char_vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2)

corpus_matrix_char = char_vectorizer.fit_transform(corpus_texts)
query_matrix_char = char_vectorizer.transform(query_texts)

print("Словарь:", len(char_vectorizer.vocabulary_))

sample_words = ['телевизор', 'телевизора', 'маникюр']
sample_vecs = char_vectorizer.transform(sample_words)
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
print(f"телевизор / телевизора: {cos_sim(sample_vecs[0], sample_vecs[1])[0][0]:.3f}")
print(f"телевизор / маникюр: {cos_sim(sample_vecs[0], sample_vecs[2])[0][0]:.3f}")

Словарь: 107635
телевизор / телевизора: 0.763
телевизор / маникюр: 0.000


In [ ]:
def evaluate_full_score(char_weight, exact_boost, near_boost, near_threshold,
                          query_matrix, corpus_matrix, query_matrix_char, corpus_matrix_char,
                          val_queries_df, item_locations_arr, corpus_ids_arr, dummy_gt,
                          batch_size=100, K=50):
    n_queries = query_matrix.shape[0]
    top_k_indices = np.zeros((n_queries, K), dtype=np.int64)
    item_locations_i32 = item_locations_arr.astype(np.int32)

    for start in range(0, n_queries, batch_size):
        end = min(start + batch_size, n_queries)

        batch_word = query_matrix[start:end]
        batch_char = query_matrix_char[start:end]
        batch_locations = val_queries_df['search_location_id'].values[start:end].astype(np.int32)

        sims_word = cosine_similarity(batch_word, corpus_matrix).astype(np.float32)
        sims_char = cosine_similarity(batch_char, corpus_matrix_char).astype(np.float32)

        sims = sims_word + char_weight * sims_char
        del sims_word, sims_char

        exact_match = (batch_locations[:, None] == item_locations_i32[None, :])
        sims += exact_boost * exact_match
        del exact_match

        diff = np.abs(batch_locations[:, None] - item_locations_i32[None, :])
        near_match = (diff > 0) & (diff <= near_threshold)
        del diff
        sims += near_boost * near_match
        del near_match

        part = np.argpartition(-sims, K, axis=1)[:, :K]
        row_indices = np.arange(sims.shape[0])[:, None]
        part_sims = sims[row_indices, part]
        order_within = np.argsort(-part_sims, axis=1)
        sorted_part = part[row_indices, order_within]
        top_k_indices[start:end] = sorted_part
        del sims

    predictions = {}
    for i, row in val_queries_df.iterrows():
        query_key = tuple(row[c] for c in query_key_cols)
        predictions[query_key] = corpus_ids_arr[top_k_indices[i]].tolist()

    return recall_at_50(predictions, dummy_gt)

for char_w in [0.0, 0.2, 0.5, 0.8]:
    r = evaluate_full_score(char_w, 1.5, 0.5, 1000,
                              query_matrix, corpus_matrix, query_matrix_char, corpus_matrix_char,
                              val_queries_df, item_locations_arr, corpus_ids_arr, dummy_gt,
                              batch_size=100)
    print(f"char_weight={char_w}: Recall@50 = {r:.4f}")

char_weight=0.0: Recall@50 = 0.5494
char_weight=0.2: Recall@50 = 0.6538
char_weight=0.5: Recall@50 = 0.6676
char_weight=0.8: Recall@50 = 0.6635


In [ ]:
for char_w in [1.0, 1.5]:
    r = evaluate_full_score(char_w, 1.5, 0.5, 1000,
                              query_matrix, corpus_matrix, query_matrix_char, corpus_matrix_char,
                              val_queries_df, item_locations_arr, corpus_ids_arr, dummy_gt,
                              batch_size=100)
    print(f"char_weight={char_w}: Recall@50 = {r:.4f}")

char_weight=1.0: Recall@50 = 0.6572
char_weight=1.5: Recall@50 = 0.6280


Лучший вес 0.5: 0.6676 — самый большой прирост после локации.

## Word2Vec

Для синонимов пробую предобученную модель RusVectōrēs `ruwikiruscorpora_upos_cbow_300_10_2021`.

In [ ]:
import gensim

MODEL_PATH = "../220/model.bin"

w2v_model = gensim.models.KeyedVectors.load_word2vec_format(MODEL_PATH, binary=True)
print("Словарь:", len(w2v_model.key_to_index))

print("маникюр в словаре:", "маникюр" in w2v_model.key_to_index)

Словарь: 249333
маникюр в словаре: False


«маникюр» в словаре нет: слова хранятся как `лемма_ЧАСТЬРЕЧИ` (`специалист_NOUN`). Перед поиском вектора слово нужно лемматизировать и добавить тег — лемма и часть речи из pymorphy2, теги переводятся в Universal POS.

In [ ]:
import pymorphy2

morph = pymorphy2.MorphAnalyzer()

POS_MAP = {
    'NOUN': 'NOUN', 'ADJF': 'ADJ', 'ADJS': 'ADJ', 'COMP': 'ADJ',
    'VERB': 'VERB', 'INFN': 'VERB', 'PRTF': 'VERB', 'PRTS': 'VERB', 'GRND': 'VERB',
    'NUMR': 'NUM', 'ADVB': 'ADV', 'NPRO': 'PRON', 'PRED': 'ADV',
    'PREP': 'ADP', 'CONJ': 'CCONJ', 'PRCL': 'PART', 'INTJ': 'INTJ',
}

def lemmatize_with_pos(text: str) -> list:
    """Возвращает список токенов вида 'лемма_POS', совместимых со словарём модели."""
    words = text.split()
    result = []
    for w in words:
        parsed = morph.parse(w)[0]
        lemma = parsed.normal_form
        pos = POS_MAP.get(parsed.tag.POS, None)
        if pos:
            result.append(f"{lemma}_{pos}")
        else:
            result.append(lemma)
    return result

print(lemmatize_with_pos("скупка телевизоров"))
print(lemmatize_with_pos("маникюр наращивание ресниц"))

['скупка_NOUN', 'телевизор_NOUN']
['маникюр_NOUN', 'наращивание_NOUN', 'ресница_NOUN']


Вектор текста — среднее векторов найденных слов.

In [ ]:
import numpy as np

EMBED_DIM = w2v_model.vector_size

def text_to_vector_pos(text: str, model, dim: int) -> np.ndarray:
    """Лемматизирует текст, добавляет POS-теги, усредняет векторы найденных токенов."""
    tokens = lemmatize_with_pos(text)
    vectors = [model[t] for t in tokens if t in model.key_to_index]
    if not vectors:
        return np.zeros(dim, dtype=np.float32)
    return np.mean(vectors, axis=0).astype(np.float32)

corpus_embed = np.zeros((len(corpus_texts), EMBED_DIM), dtype=np.float32)
for i, text in enumerate(corpus_texts):
    corpus_embed[i] = text_to_vector_pos(text, w2v_model, EMBED_DIM)

query_embed = np.zeros((len(query_texts), EMBED_DIM), dtype=np.float32)
for i, text in enumerate(query_texts):
    query_embed[i] = text_to_vector_pos(text, w2v_model, EMBED_DIM)

zero_corpus = (corpus_embed.sum(axis=1) == 0).sum()
zero_query = (query_embed.sum(axis=1) == 0).sum()
print(f"Нулевых векторов в корпусе: {zero_corpus} ({zero_corpus/len(corpus_texts)*100:.2f}%)")
print(f"Нулевых векторов в запросах: {zero_query} ({zero_query/len(query_texts)*100:.2f}%)")

Нулевых векторов в корпусе: 3548 (1.03%)
Нулевых векторов в запросах: 1277 (4.03%)


In [ ]:
from sklearn.preprocessing import normalize

corpus_embed_norm = normalize(corpus_embed, axis=1)
query_embed_norm = normalize(query_embed, axis=1)

corpus_embed_norm = np.nan_to_num(corpus_embed_norm, nan=0.0)
query_embed_norm = np.nan_to_num(query_embed_norm, nan=0.0)

Пересчитывать всё для каждого веса слишком долго, поэтому тяжёлую часть считаю один раз: для каждого запроса сохраняю топ-300 кандидатов со скором word + char + локация и отдельно скором Word2Vec. Дальше веса перебираются на маленькой матрице 31673 × 300.

In [ ]:
def compute_base_and_embed_sims(char_weight, exact_boost, near_boost, near_threshold,
                                   query_matrix, corpus_matrix, query_matrix_char, corpus_matrix_char,
                                   query_embed_norm, corpus_embed_norm,
                                   val_queries_df, item_locations_arr,
                                   batch_size=100):
    n_queries = query_matrix.shape[0]
    item_locations_i32 = item_locations_arr.astype(np.int32)

    M = 300

    candidates_idx = np.zeros((n_queries, M), dtype=np.int32)
    candidates_base = np.zeros((n_queries, M), dtype=np.float32)
    candidates_embed = np.zeros((n_queries, M), dtype=np.float32)

    for start in range(0, n_queries, batch_size):
        end = min(start + batch_size, n_queries)

        batch_word = query_matrix[start:end]
        batch_char = query_matrix_char[start:end]
        batch_embed = query_embed_norm[start:end]
        batch_locations = val_queries_df['search_location_id'].values[start:end].astype(np.int32)

        sims_word = cosine_similarity(batch_word, corpus_matrix).astype(np.float32)
        sims_char = cosine_similarity(batch_char, corpus_matrix_char).astype(np.float32)
        sims_embed = (batch_embed @ corpus_embed_norm.T).astype(np.float32)

        base = sims_word + char_weight * sims_char
        del sims_word, sims_char

        exact_match = (batch_locations[:, None] == item_locations_i32[None, :])
        base += exact_boost * exact_match
        del exact_match

        diff = np.abs(batch_locations[:, None] - item_locations_i32[None, :])
        near_match = (diff > 0) & (diff <= near_threshold)
        del diff
        base += near_boost * near_match
        del near_match

        combined_for_pooling = base + 0.5 * sims_embed
        part = np.argpartition(-combined_for_pooling, M, axis=1)[:, :M]
        del combined_for_pooling

        row_idx = np.arange(base.shape[0])[:, None]
        candidates_idx[start:end] = part
        candidates_base[start:end] = base[row_idx, part]
        candidates_embed[start:end] = sims_embed[row_idx, part]

        del base, sims_embed

    return candidates_idx, candidates_base, candidates_embed

cand_idx, cand_base, cand_embed = compute_base_and_embed_sims(
    0.5, 1.5, 0.5, 1000,
    query_matrix, corpus_matrix, query_matrix_char, corpus_matrix_char,
    query_embed_norm, corpus_embed_norm,
    val_queries_df, item_locations_arr, batch_size=100
)

In [ ]:
K = 50

def recall_from_pool(embed_weight, cand_idx, cand_base, cand_embed, corpus_ids_arr, val_queries_df, dummy_gt, K=50):
    combined = cand_base + embed_weight * cand_embed
    order = np.argsort(-combined, axis=1)[:, :K]

    row_idx = np.arange(cand_idx.shape[0])[:, None]
    top_k_idx = cand_idx[row_idx, order]

    predictions = {}
    for i, row in val_queries_df.iterrows():
        query_key = tuple(row[c] for c in query_key_cols)
        predictions[query_key] = corpus_ids_arr[top_k_idx[i]].tolist()

    return recall_at_50(predictions, dummy_gt)

r_sanity = recall_from_pool(0.0, cand_idx, cand_base, cand_embed, corpus_ids_arr, val_queries_df, dummy_gt)

for embed_w in [0.0, 0.1, 0.3, 0.5, 0.8, 1.2, 2.0]:
    r = recall_from_pool(embed_w, cand_idx, cand_base, cand_embed, corpus_ids_arr, val_queries_df, dummy_gt)
    print(f"embed_weight={embed_w}: Recall@50 = {r:.4f}")

embed_weight=0.0: Recall@50 = 0.6678
embed_weight=0.1: Recall@50 = 0.6783
embed_weight=0.3: Recall@50 = 0.6768
embed_weight=0.5: Recall@50 = 0.6729
embed_weight=0.8: Recall@50 = 0.6622
embed_weight=1.2: Recall@50 = 0.6453
embed_weight=2.0: Recall@50 = 0.6127


Word2Vec добавляет немного: +0.01 при весе 0.1, дальше хуже.

## BM25

`rank_bm25` по заголовкам.

In [ ]:
from rank_bm25 import BM25Okapi

corpus_tokens = [text.split() for text in corpus_texts]

bm25 = BM25Okapi(corpus_tokens)

sample_query_tokens = query_texts[0].split()
sample_scores = bm25.get_scores(sample_query_tokens)
print("Запрос:", query_texts[0])
print("Скоры топ-5:", sample_scores[np.argsort(-sample_scores)[:5]])

Запрос: скупка телевизоров
Скоры топ-5: [13.17662641 13.17662641 13.17662641 13.17662641 13.17662641]


У первых пяти документов одинаковый скор. Это не ошибка: в корпусе несколько объявлений с заголовком ровно «Скупка телевизоров», а одинаковый текст даёт одинаковый скор.

In [ ]:
import time
start_time = time.time()
_ = bm25.get_scores(sample_query_tokens)
elapsed = time.time() - start_time
print(f"Время на один запрос: {elapsed:.3f} сек")
print(f"Оценка времени на все 31673 запроса: {elapsed * 31673 / 60:.1f} минут")

Время на один запрос: 0.091 сек
Оценка времени на все 31673 запроса: 48.2 минут


0.09 с на запрос — около 48 минут на валидацию, для перебора весов слишком медленно. В пайплайне BM25 посчитан матрично через разреженные матрицы, это занимает около минуты.

## Итог

| шаг | Recall@50 |
|---|---|
| TF-IDF по заголовку | 0.155 |
| + бонус за локацию | 0.528 |
| + бонус за близкий `location_id` | 0.549 |
| + символьные n-граммы | 0.668 |
| + Word2Vec | 0.678 |

Дальше работа перенесена в `src/avito_cg`: BM25 по заголовку вместе с описанием, e5 вместо Word2Vec, расстояние по координатам вместо близости ID, микрокатегории по похожим запросам и общий пул кандидатов. Там же изменена валидация — один запрос на текст, как в бенчмарке, — поэтому цифры напрямую с этой таблицей не сравниваются.